In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd /content/drive/My Drive/Machine Learning/WaferDetect

In [ ]:
!unzip -q "/content/drive/My Drive/Machine Learning/WaferDetect/waferdetect.zip" -d "/content/drive/My Drive/Machine Learning/WaferDetect"

In [ ]:
!pip install -q uv
!uv sync

In [ ]:
!uv run pytest -q

In [ ]:
!uv run python -m scripts.perception.dataset --force

In [ ]:
!uv run python -m scripts.perception.train --device 0

In [ ]:
!uv run python -m scripts.perception.evaluate --model-path /content/waferdetect_runs/train/stage1_baseline/weights/best.pt --name stage1_baseline

In [ ]:
!zip -r /content/waferdetect_runs.zip /content/waferdetect_runs

In [ ]:
!uv run python -m scripts.datagen.generator --out-dir data/generated/v1 --count 10000 --seed 42 --physics-frac 0.5

In [ ]:
%%bash
for N in 500 1000 2000 5000 10000; do
    uv run python -m scripts.datagen.layout --generated-dir data/generated/v1 --out-dir data/generated/v1_yolo_${N} --limit ${N}
    uv run python -m scripts.perception.train --data data/generated/v1_yolo_${N}/data.yaml --name stage2_scale_${N} --device 0
    uv run python -m scripts.perception.evaluate --model-path runs/train/stage2_scale_${N}/weights/best.pt --data data/generated/v1_yolo_${N}/data.yaml --name stage2_scale_${N}
done

In [ ]:
!uv run python -m scripts.datagen.generator --out-dir data/generated/v1_quant --count 10000 --seed 42 --physics-frac 0.5 --die-grid-frac 0.5
!uv run python -m scripts.datagen.layout --generated-dir data/generated/v1_quant --out-dir data/generated/v1_quant_yolo
!uv run python -m scripts.perception.train --data data/generated/v1_quant_yolo/data.yaml --name stage2_quant --device 0

In [ ]:
!uv run python -m scripts.baselines.classical